# Modul 6: Integrasi Teknik AI dalam Sistem Informasi
## Studi Kasus 3: Sistem Informasi Penerimaan Mahasiswa Baru (Prediksi Kemungkinan Lolos Seleksi)

### Deskripsi Studi Kasus & Arsitektur Komponen
Notebook ini mengimplementasikan arsitektur berlapis (*Layered Architecture*) untuk memprediksi peluang kelulusan calon mahasiswa (`0` = Tidak Lolos Seleksi, `1` = Lolos Seleksi) berdasarkan komponen penilaian seleksi:
1. **Model Layer (Fase Training & Serialisasi):** Melatih `MLPClassifier` dengan penyetaraan skala fitur (`StandardScaler`) dan menyimpannya ke `model_seleksi_pmb.joblib`.
2. **Data Layer (SQLite3):** Mencatat riwayat input penilaian pendaftar, hasil prediksi, dan stempel waktu ke tabel `log_seleksi` pada `log_pmb.db`.
3. **Application Layer (`templates/index.html`):** Antarmuka web bagi panitia seleksi PMB atau calon pendaftar.
4. **API Layer (Flask):** Menyediakan endpoint REST `/predict` (POST) untuk memvalidasi skor (0–100) dan mengembalikan prediksi kelulusan secara *real-time* (Fase Serving).
5. **Pengujian End-to-End:** Menguji endpoint pada beberapa profil pendaftar menggunakan `requests` dan memverifikasi tabel log SQLite.

### 1. Instalasi & Impor Pustaka

In [1]:
# Jalankan baris di bawah jika pustaka belum terpasang:
# !pip install scikit-learn joblib flask requests numpy

import os
import sqlite3
import threading
import time
from datetime import datetime
import joblib
import numpy as np
import requests
from flask import Flask, jsonify, render_template, request
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

### 2. Model Layer: Pelatihan & Serialisasi Model (`train_model.py`)
* **Fitur Input:** `[nilai_rapor, skor_tes, nilai_wawancara]` (skala `0` – `100`)
* **Target Label:** `0` = Tidak Lolos Seleksi, `1` = Lolos Seleksi

In [2]:
# Dataset historis pendaftar PMB tahun-tahun sebelumnya
# Fitur: [nilai_rapor, skor_tes_tertulis, nilai_wawancara] (skala 0 - 100)
X = np.array([
    [88.5, 85.0, 90.0],  # Semua komponen tinggi -> Lolos (1)
    [91.0, 82.0, 86.0],  # Rapor unggul, tes & wawancara baik -> Lolos (1)
    [65.0, 60.0, 68.0],  # Semua komponen rendah -> Tidak Lolos (0)
    [70.0, 58.0, 62.0],  # Skor tes rendah -> Tidak Lolos (0)
    [84.0, 80.0, 82.0],  # Konsisten di atas ambang batas -> Lolos (1)
    [62.0, 64.0, 55.0],  # Wawancara & rapor rendah -> Tidak Lolos (0)
    [89.0, 88.0, 85.0],  # Kandidat kuat -> Lolos (1)
    [72.0, 65.0, 60.0],  # Belum memenuhi standar -> Tidak Lolos (0)
    [82.0, 79.0, 84.0],  # Wawancara kuat menopang nilai cukup -> Lolos (1)
    [68.0, 62.0, 65.0],  # Rata-rata di bawah standar -> Tidak Lolos (0)
    [93.0, 90.0, 88.0],  # Kandidat sangat unggul -> Lolos (1)
    [74.0, 61.0, 64.0],  # Skor tes & wawancara kurang -> Tidak Lolos (0)
])

y = np.array([1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0])

# Membagi data latih dan data uji (75% latih, 25% uji) dengan stratifikasi kelas
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Pipeline: StandardScaler + MLPClassifier
model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(8, 4), activation="relu", max_iter=2000, random_state=42)
)

# Melatih model
model.fit(X_train, y_train)
print(f"Akurasi model pada data uji: {model.score(X_test, y_test):.2f}")

# Serialisasi model ke file fisik
nama_file_model = "model_seleksi_pmb.joblib"
joblib.dump(model, nama_file_model)
print(f"Model berhasil disimpan sebagai '{nama_file_model}'")

Akurasi model pada data uji: 1.00
Model berhasil disimpan sebagai 'model_seleksi_pmb.joblib'


/opt/anaconda/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(


### 3. Application Layer: Membuat Antarmuka Web (`templates/index.html`)

In [3]:
os.makedirs("templates", exist_ok=True)

html_content = '''<!DOCTYPE html>
<html lang="id">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Prediksi Kelulusan Seleksi - PMB</title>
    <style>
        body { font-family: sans-serif; max-width: 480px; margin: 40px auto; line-height: 1.6; }
        .form-group { margin-bottom: 14px; }
        label { display: block; font-weight: bold; margin-bottom: 4px; }
        input, button { width: 100%; padding: 8px; box-sizing: border-box; }
        button { background: #111; color: #fff; border: none; cursor: pointer; font-weight: bold; margin-top: 8px; }
        button:hover { background: #333; }
        #hasil { margin-top: 20px; padding: 12px; border: 1px solid #ccc; display: none; }
    </style>
</head>
<body>
    <h2>Prediksi Kelulusan Penerimaan Mahasiswa Baru</h2>
    <p>Masukkan komponen nilai seleksi calon mahasiswa (skala 0 - 100).</p>

    <div class="form-group">
        <label for="nilai_rapor">Rata-Rata Nilai Rapor (0 - 100):</label>
        <input type="number" id="nilai_rapor" step="0.1" min="0" max="100" placeholder="Contoh: 86.5" required>
    </div>

    <div class="form-group">
        <label for="skor_tes">Skor Tes Tertulis (0 - 100):</label>
        <input type="number" id="skor_tes" step="0.1" min="0" max="100" placeholder="Contoh: 82.0" required>
    </div>

    <div class="form-group">
        <label for="nilai_wawancara">Nilai Wawancara (0 - 100):</label>
        <input type="number" id="nilai_wawancara" step="0.1" min="0" max="100" placeholder="Contoh: 85.0" required>
    </div>

    <button onclick="prediksiKelulusan()">Prediksi Kelulusan</button>

    <div id="hasil"></div>

    <script>
        async function prediksiKelulusan() {
            const hasilDiv = document.getElementById("hasil");
            const payload = {
                nilai_rapor: parseFloat(document.getElementById("nilai_rapor").value),
                skor_tes: parseFloat(document.getElementById("skor_tes").value),
                nilai_wawancara: parseFloat(document.getElementById("nilai_wawancara").value)
            };

            try {
                const response = await fetch("/predict", {
                    method: "POST",
                    headers: { "Content-Type": "application/json" },
                    body: JSON.stringify(payload)
                });

                const data = await response.json();
                hasilDiv.style.display = "block";

                if (response.ok) {
                    hasilDiv.innerHTML = `
                        <strong>Hasil Prediksi:</strong> ${data.label}<br>
                        <strong>Tingkat Keyakinan Model:</strong> ${(data.probabilitas * 100).toFixed(1)}%
                    `;
                } else {
                    hasilDiv.innerHTML = `<strong>Error:</strong> ${data.error}`;
                }
            } catch (err) {
                hasilDiv.style.display = "block";
                hasilDiv.innerHTML = `<strong>Gagal terhubung ke API:</strong> ${err.message}`;
            }
        }
    </script>
</body>
</html>'''

with open("templates/index.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("File 'templates/index.html' berhasil dibuat.")

File 'templates/index.html' berhasil dibuat.


### 4. API Layer & Data Layer (`app.py`)

In [4]:
app = Flask(__name__)
DB_NAME = "log_pmb.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS log_seleksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            waktu TEXT NOT NULL,
            bulan_tahun TEXT NOT NULL,
            nilai_rapor REAL NOT NULL,
            skor_tes REAL NOT NULL,
            nilai_wawancara REAL NOT NULL,
            prediksi INTEGER NOT NULL,
            label TEXT NOT NULL,
            probabilitas REAL NOT NULL
        )
    ''')
    conn.commit()
    conn.close()

def log_prediction(data, prediksi, label, probabilitas):
    sekarang = datetime.now()
    conn = sqlite3.connect(DB_NAME)
    conn.execute(
        '''
        INSERT INTO log_seleksi (
            waktu, bulan_tahun, nilai_rapor, skor_tes,
            nilai_wawancara, prediksi, label, probabilitas
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''',
        (
            sekarang.isoformat(),
            sekarang.strftime("%Y-%m"),
            data["nilai_rapor"],
            data["skor_tes"],
            data["nilai_wawancara"],
            prediksi,
            label,
            probabilitas,
        ),
    )
    conn.commit()
    conn.close()

init_db()
loaded_model = joblib.load("model_seleksi_pmb.joblib")

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    if not data:
        return jsonify({"error": "Payload JSON tidak ditemukan"}), 400

    try:
        nilai_rapor = float(data["nilai_rapor"])
        skor_tes = float(data["skor_tes"])
        nilai_wawancara = float(data["nilai_wawancara"])

        for nama, nilai in [
            ("nilai_rapor", nilai_rapor),
            ("skor_tes", skor_tes),
            ("nilai_wawancara", nilai_wawancara),
        ]:
            if not (0 <= nilai <= 100):
                return jsonify({"error": f"{nama} harus berada pada rentang 0 - 100"}), 400

        fitur = np.array([[nilai_rapor, skor_tes, nilai_wawancara]])
    except KeyError as e:
        return jsonify({"error": f"Field wajib hilang: {e}"}), 400
    except (ValueError, TypeError):
        return jsonify({"error": "Tipe data input harus berupa angka"}), 400

    prediksi = int(loaded_model.predict(fitur)[0])
    probabilitas = float(loaded_model.predict_proba(fitur)[0][prediksi])
    label = "Lolos Seleksi" if prediksi == 1 else "Tidak Lolos Seleksi"

    log_prediction(
        {
            "nilai_rapor": nilai_rapor,
            "skor_tes": skor_tes,
            "nilai_wawancara": nilai_wawancara,
        },
        prediksi,
        label,
        round(probabilitas, 3),
    )

    return jsonify({
        "prediksi": prediksi,
        "label": label,
        "probabilitas": round(probabilitas, 3),
    })

print("Aplikasi Flask & SQLite siap.")

Aplikasi Flask & SQLite siap.


### 5. Menjalankan Server & Pengujian End-to-End (`test_api.py`)

In [6]:
PORT = 5003

def run_server():
    app.run(port=PORT, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)

URL_API = f"http://127.0.0.1:{PORT}/predict"

test_cases = [
    {"nilai_rapor": 90.0, "skor_tes": 86.0, "nilai_wawancara": 88.0},  # Kandidat kuat -> Lolos
    {"nilai_rapor": 64.0, "skor_tes": 59.0, "nilai_wawancara": 61.0},  # Nilai rendah -> Tidak Lolos
    {"nilai_rapor": 83.0, "skor_tes": 80.0, "nilai_wawancara": 85.0},  # Kandidat kompetitif -> Lolos
    {"nilai_rapor": 71.0, "skor_tes": 63.0, "nilai_wawancara": 60.0},  # Di bawah ambang -> Tidak Lolos
]

print("=== 1. MENGUJI ENDPOINT REST API ===")
for idx, kasus in enumerate(test_cases, start=1):
    res = requests.post(URL_API, json=kasus)
    print(f"Kasus {idx}: Input={kasus} -> Status={res.status_code}, Output={res.json()}")

print("=== 2. VERIFIKASI DATA LAYER (SQLITE LOG) ===")
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()
cursor.execute("SELECT id, waktu, bulan_tahun, nilai_rapor, skor_tes, nilai_wawancara, label, probabilitas FROM log_seleksi")
rows = cursor.fetchall()
conn.close()

print(f"Total baris tercatat di database: {len(rows)}")
for row in rows[-len(test_cases):]:
    print(row)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5003
Press CTRL+C to quit
127.0.0.1 - - [19/Nov/2026 19:20:16] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 19:20:16] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 19:20:16] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 19:20:16] "POST /predict HTTP/1.1" 200 -


=== 1. MENGUJI ENDPOINT REST API ===
Kasus 1: Input={'nilai_rapor': 90.0, 'skor_tes': 86.0, 'nilai_wawancara': 88.0} -> Status=200, Output={'label': 'Lolos Seleksi', 'prediksi': 1, 'probabilitas': 0.661}
Kasus 2: Input={'nilai_rapor': 64.0, 'skor_tes': 59.0, 'nilai_wawancara': 61.0} -> Status=200, Output={'label': 'Tidak Lolos Seleksi', 'prediksi': 0, 'probabilitas': 1.0}
Kasus 3: Input={'nilai_rapor': 83.0, 'skor_tes': 80.0, 'nilai_wawancara': 85.0} -> Status=200, Output={'label': 'Lolos Seleksi', 'prediksi': 1, 'probabilitas': 0.652}
Kasus 4: Input={'nilai_rapor': 71.0, 'skor_tes': 63.0, 'nilai_wawancara': 60.0} -> Status=200, Output={'label': 'Tidak Lolos Seleksi', 'prediksi': 0, 'probabilitas': 1.0}
=== 2. VERIFIKASI DATA LAYER (SQLITE LOG) ===
Total baris tercatat di database: 4
(1, '2026-11-19T19:20:16.499370', '2026-11', 90.0, 86.0, 88.0, 'Lolos Seleksi', 0.661)
(2, '2026-11-19T19:20:16.517620', '2026-11', 64.0, 59.0, 61.0, 'Tidak Lolos Seleksi', 1.0)
(3, '2026-11-19T19:20:16.53

127.0.0.1 - - [19/Nov/2026 19:20:24] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Nov/2026 19:20:24] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [19/Nov/2026 19:20:43] "POST /predict HTTP/1.1" 200 -


### 6. Jawaban Pertanyaan Latihan Mandiri

1. **Fitur Input Model & Relevansinya:**
   * `nilai_rapor`: Mencerminkan konsistensi capaian akademik jangka panjang calon mahasiswa di sekolah menengah.
   * `skor_tes`: Mengukur kemampuan kognitif, penalaran logis, dan penguasaan materi dasar secara terstandar antar pendaftar.
   * `nilai_wawancara`: Menilai aspek non-kognitif seperti motivasi studi, komunikasi, dan kesesuaian minat program studi.
2. **Perancangan Skema Tabel Log untuk Analisis Lanjutan:**
   * Tabel `log_seleksi` menyimpan `waktu` (ISO-8601) dan `bulan_tahun` (`YYYY-MM`) untuk mempermudah analisis rasio kelulusan lintas gelombang/bulan pendaftaran (`GROUP BY bulan_tahun, label`), serta kolom `probabilitas` untuk mengidentifikasi kasus ambang batas (*borderline*) yang butuh tinjauan manual.
3. **Risiko Jika API Model Tidak Tersedia (*Down*) & Solusinya:**
   * **Risiko:** Pada puncak masa pendaftaran, kegagalan layanan API dapat menghentikan proses pengajuan berkas atau simulasi kelulusan pada portal PMB.
   * **Mitigasi:** Menerapkan *graceful degradation* (fallback ke rumus pembobotan nilai statis jika koneksi API gagal), memisahkan penyimpanan berkas dengan pemrosesan prediksi menggunakan antrean asinkron (*message queue*), serta menjalankan beberapa worker API dengan *load balancing*.